# Módulo de Configuración Histórica
## Notebook 03 — Datos de Ejemplo & Consultas de Validación

Este notebook:
1. Inserta un conjunto de datos de ejemplo realista
2. Ejecuta consultas de validación que recorren toda la cadena de relaciones

### Escenario de ejemplo
María García trabajó como **Asesora Comercial** de enero a junio 2024, y a partir de julio  
se convirtió en **Líder Regional**. Cada rol tuvo su propio esquema de compensación con  
evaluaciones y metas diferentes.

> **Prerrequisito**: Ejecuta primero **01_dimensiones** y **02_asignaciones_y_esquemas**.

---
## 0 · Configuración

In [ ]:
LAKEHOUSE_NAME = "BI - Bandelta LH"

spark.sql(f"USE `{LAKEHOUSE_NAME}`")
print(f"✅ Usando Lakehouse: {LAKEHOUSE_NAME}")

---
## 1 · Insertar catálogos de ejemplo

### 1.1 dim_usuario

In [ ]:
from pyspark.sql import Row
from datetime import date, datetime

# ─── Insertar solo si está vacía ──────────────────────────────────────────────
if spark.sql("SELECT COUNT(*) as c FROM dim_usuario").collect()[0].c == 0:
    usuarios = [
        Row(id_usuario=1, nombre="María García",   email="m.garcia@empresa.mx",
            codigo_empleado="EMP-001", area="Comercial",       activo=True,
            fecha_creacion=datetime(2023, 1, 1), fecha_baja=None),
        Row(id_usuario=2, nombre="Carlos López",   email="c.lopez@empresa.mx",
            codigo_empleado="EMP-002", area="Comercial",       activo=True,
            fecha_creacion=datetime(2023, 1, 1), fecha_baja=None),
        Row(id_usuario=3, nombre="Ana Torres",     email="a.torres@empresa.mx",
            codigo_empleado="EMP-003", area="Comercial Sur",   activo=True,
            fecha_creacion=datetime(2023, 6, 1), fecha_baja=None),
    ]
    spark.createDataFrame(usuarios).write.mode("append").saveAsTable("dim_usuario")
    print("✅ 3 usuarios insertados.")
else:
    print("ℹ️  dim_usuario ya tiene datos.")

spark.sql("SELECT * FROM dim_usuario").show(truncate=False)

### 1.2 dim_rol

In [ ]:
if spark.sql("SELECT COUNT(*) as c FROM dim_rol").collect()[0].c == 0:
    roles = [
        Row(id_rol=1, nombre_rol="Asesor Comercial",  descripcion="Atención directa a clientes y cierre de ventas",
            nivel="Operativo",   activo=True),
        Row(id_rol=2, nombre_rol="Líder Regional",    descripcion="Gestión de equipo comercial en una región",
            nivel="Táctico",     activo=True),
        Row(id_rol=3, nombre_rol="Director Comercial",descripcion="Estrategia comercial nacional",
            nivel="Estratégico", activo=True),
    ]
    spark.createDataFrame(roles).write.mode("append").saveAsTable("dim_rol")
    print("✅ 3 roles insertados.")
else:
    print("ℹ️  dim_rol ya tiene datos.")

spark.sql("SELECT * FROM dim_rol").show(truncate=False)

### 1.3 dim_evaluacion

In [ ]:
if spark.sql("SELECT COUNT(*) as c FROM dim_evaluacion").collect()[0].c == 0:
    evaluaciones = [
        Row(id_evaluacion=1, nombre_evaluacion="Ventas Netas",
            descripcion="Monto total de ventas menos devoluciones",
            tipo_metrica="valor_absoluto", unidad_medida="MXN",   activo=True),
        Row(id_evaluacion=2, nombre_evaluacion="Clientes Nuevos",
            descripcion="Número de clientes adquiridos en el período",
            tipo_metrica="conteo",         unidad_medida="clientes", activo=True),
        Row(id_evaluacion=3, nombre_evaluacion="Tasa de Retención",
            descripcion="Porcentaje de clientes que renovaron",
            tipo_metrica="porcentaje",     unidad_medida="%",      activo=True),
        Row(id_evaluacion=4, nombre_evaluacion="Productividad del Equipo",
            descripcion="Ventas promedio del equipo que gestiona",
            tipo_metrica="valor_absoluto", unidad_medida="MXN",   activo=True),
    ]
    spark.createDataFrame(evaluaciones).write.mode("append").saveAsTable("dim_evaluacion")
    print("✅ 4 evaluaciones insertadas.")
else:
    print("ℹ️  dim_evaluacion ya tiene datos.")

spark.sql("SELECT * FROM dim_evaluacion").show(truncate=False)

---
## 2 · Insertar asignaciones versionadas

### 2.1 usuario_rol_historico — María cambia de rol en julio 2024

In [ ]:
if spark.sql("SELECT COUNT(*) as c FROM usuario_rol_historico").collect()[0].c == 0:
    asignaciones_usuario = [
        # María: Asesora Comercial ene–jun 2024
        Row(id_asignacion=1, id_usuario=1, id_rol=1,
            fecha_inicio=date(2024, 1, 1), fecha_fin=date(2024, 6, 30),
            activo=True, fecha_creacion=datetime(2024, 1, 2),
            usuario_creacion="admin"),
        # María: Líder Regional jul 2024 en adelante
        Row(id_asignacion=2, id_usuario=1, id_rol=2,
            fecha_inicio=date(2024, 7, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 7, 1),
            usuario_creacion="admin"),
        # Carlos: Asesor Comercial desde ene 2024, sigue activo
        Row(id_asignacion=3, id_usuario=2, id_rol=1,
            fecha_inicio=date(2024, 1, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 1, 2),
            usuario_creacion="admin"),
        # Ana: Asesora Comercial desde jul 2024
        Row(id_asignacion=4, id_usuario=3, id_rol=1,
            fecha_inicio=date(2024, 7, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 7, 1),
            usuario_creacion="admin"),
    ]
    spark.createDataFrame(asignaciones_usuario).write.mode("append").saveAsTable("usuario_rol_historico")
    print("✅ Asignaciones usuario-rol insertadas.")
else:
    print("ℹ️  usuario_rol_historico ya tiene datos.")

spark.sql("""
    SELECT urh.*, u.nombre, r.nombre_rol
    FROM usuario_rol_historico urh
    JOIN dim_usuario u ON u.id_usuario = urh.id_usuario
    JOIN dim_rol r     ON r.id_rol     = urh.id_rol
    ORDER BY urh.id_usuario, urh.fecha_inicio
""").show(truncate=False)

### 2.2 rol_esquema_historico — cada rol con su esquema de compensación

In [ ]:
if spark.sql("SELECT COUNT(*) as c FROM rol_esquema_historico").collect()[0].c == 0:
    esquemas = [
        # Asesor Comercial — esquema 2024
        Row(id_asignacion_esquema=1, id_rol=1, id_esquema=101,
            nombre_esquema="Esquema Asesor 2024",
            fecha_inicio=date(2024, 1, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 1, 1),
            usuario_creacion="admin"),
        # Líder Regional — esquema 2024
        Row(id_asignacion_esquema=2, id_rol=2, id_esquema=201,
            nombre_esquema="Esquema Líder 2024",
            fecha_inicio=date(2024, 1, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 1, 1),
            usuario_creacion="admin"),
    ]
    spark.createDataFrame(esquemas).write.mode("append").saveAsTable("rol_esquema_historico")
    print("✅ Esquemas de rol insertados.")
else:
    print("ℹ️  rol_esquema_historico ya tiene datos.")

spark.sql("""
    SELECT reh.*, r.nombre_rol
    FROM rol_esquema_historico reh
    JOIN dim_rol r ON r.id_rol = reh.id_rol
    ORDER BY reh.id_rol
""").show(truncate=False)

---
## 3 · Insertar detalle de esquemas

### 3.1 esquema_evaluacion_historico — pesos por evaluación

- **Asesor (esquema 101)**: Ventas 50% + Clientes Nuevos 30% + Retención 20%
- **Líder (esquema 201)**: Productividad del Equipo 40% + Ventas 35% + Retención 25%

In [ ]:
from decimal import Decimal

if spark.sql("SELECT COUNT(*) as c FROM esquema_evaluacion_historico").collect()[0].c == 0:
    detalles = [
        # Esquema 101 — Asesor Comercial
        Row(id_detalle=1,  id_esquema=101, id_evaluacion=1, peso=Decimal("0.5000"),
            fecha_inicio=date(2024, 1, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 1, 1)),
        Row(id_detalle=2,  id_esquema=101, id_evaluacion=2, peso=Decimal("0.3000"),
            fecha_inicio=date(2024, 1, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 1, 1)),
        Row(id_detalle=3,  id_esquema=101, id_evaluacion=3, peso=Decimal("0.2000"),
            fecha_inicio=date(2024, 1, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 1, 1)),
        # Esquema 201 — Líder Regional
        Row(id_detalle=4,  id_esquema=201, id_evaluacion=4, peso=Decimal("0.4000"),
            fecha_inicio=date(2024, 1, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 1, 1)),
        Row(id_detalle=5,  id_esquema=201, id_evaluacion=1, peso=Decimal("0.3500"),
            fecha_inicio=date(2024, 1, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 1, 1)),
        Row(id_detalle=6,  id_esquema=201, id_evaluacion=3, peso=Decimal("0.2500"),
            fecha_inicio=date(2024, 1, 1), fecha_fin=None,
            activo=True, fecha_creacion=datetime(2024, 1, 1)),
    ]
    spark.createDataFrame(detalles).write.mode("append").saveAsTable("esquema_evaluacion_historico")
    print("✅ Pesos de evaluación insertados.")
else:
    print("ℹ️  esquema_evaluacion_historico ya tiene datos.")

# Verificar que la suma de pesos sea 1.0 por esquema
spark.sql("""
    SELECT id_esquema, ROUND(SUM(peso), 4) AS suma_pesos,
           CASE WHEN ROUND(SUM(peso), 4) = 1.0 THEN '✅ OK' ELSE '❌ Error' END AS validacion
    FROM esquema_evaluacion_historico
    WHERE activo = TRUE AND fecha_fin IS NULL
    GROUP BY id_esquema
""").show()

### 3.2 meta_evaluacion_historico — metas por período

In [ ]:
if spark.sql("SELECT COUNT(*) as c FROM meta_evaluacion_historico").collect()[0].c == 0:

    # Obtener IDs de periodos ene–ago 2024
    periodos_2024 = spark.sql("""
        SELECT id_periodo, mes FROM dim_periodo
        WHERE anio = 2024 AND mes BETWEEN 1 AND 8
        ORDER BY mes
    """).collect()

    metas = []
    mid = 1

    # Metas esquema 101 (Asesor Comercial) — ene a ago 2024
    # Ventas escalando mensualmente, Clientes Nuevos fijo, Retención fijo
    ventas_base    = [80_000, 85_000, 90_000, 90_000, 95_000, 95_000, 100_000, 100_000]
    clientes_base  = [5, 5, 6, 6, 7, 7, 7, 8]
    retencion_base = [85, 85, 87, 87, 88, 88, 88, 90]   # porcentaje

    for i, row in enumerate(periodos_2024):
        pid = row.id_periodo
        metas += [
            Row(id_meta=mid,   id_esquema=101, id_evaluacion=1, id_periodo=pid,
                valor_meta=Decimal(str(ventas_base[i])),
                valor_minimo=Decimal(str(ventas_base[i] * 0.8)),
                valor_maximo=Decimal(str(ventas_base[i] * 1.2)),
                fecha_creacion=datetime(2024, 1, 1), usuario_creacion="admin"),
            Row(id_meta=mid+1, id_esquema=101, id_evaluacion=2, id_periodo=pid,
                valor_meta=Decimal(str(clientes_base[i])),
                valor_minimo=Decimal(str(clientes_base[i] - 1)),
                valor_maximo=None,
                fecha_creacion=datetime(2024, 1, 1), usuario_creacion="admin"),
            Row(id_meta=mid+2, id_esquema=101, id_evaluacion=3, id_periodo=pid,
                valor_meta=Decimal(str(retencion_base[i])),
                valor_minimo=Decimal("80"),
                valor_maximo=Decimal("100"),
                fecha_creacion=datetime(2024, 1, 1), usuario_creacion="admin"),
        ]
        mid += 3

    # Metas esquema 201 (Líder Regional) — ene a ago 2024
    productividad_base = [400_000, 420_000, 440_000, 440_000, 460_000, 460_000, 480_000, 480_000]
    ventas_lider       = [200_000, 210_000, 220_000, 220_000, 230_000, 230_000, 240_000, 240_000]
    retencion_lider    = [88, 88, 89, 89, 90, 90, 90, 92]

    for i, row in enumerate(periodos_2024):
        pid = row.id_periodo
        metas += [
            Row(id_meta=mid,   id_esquema=201, id_evaluacion=4, id_periodo=pid,
                valor_meta=Decimal(str(productividad_base[i])),
                valor_minimo=Decimal(str(productividad_base[i] * 0.85)),
                valor_maximo=None,
                fecha_creacion=datetime(2024, 1, 1), usuario_creacion="admin"),
            Row(id_meta=mid+1, id_esquema=201, id_evaluacion=1, id_periodo=pid,
                valor_meta=Decimal(str(ventas_lider[i])),
                valor_minimo=Decimal(str(ventas_lider[i] * 0.8)),
                valor_maximo=None,
                fecha_creacion=datetime(2024, 1, 1), usuario_creacion="admin"),
            Row(id_meta=mid+2, id_esquema=201, id_evaluacion=3, id_periodo=pid,
                valor_meta=Decimal(str(retencion_lider[i])),
                valor_minimo=Decimal("85"),
                valor_maximo=Decimal("100"),
                fecha_creacion=datetime(2024, 1, 1), usuario_creacion="admin"),
        ]
        mid += 3

    spark.createDataFrame(metas).write.mode("append").saveAsTable("meta_evaluacion_historico")
    print(f"✅ {len(metas)} metas insertadas.")
else:
    print("ℹ️  meta_evaluacion_historico ya tiene datos.")

---
## 4 · Consultas de Validación

### 4.1 ¿Qué rol tenía María en un mes específico?

In [ ]:
# ─── Cambia la fecha para explorar distintos escenarios ──────────────────────
FECHA_CONSULTA = "2024-03-15"   # María debería ser Asesora Comercial
# FECHA_CONSULTA = "2024-09-01"  # María debería ser Líder Regional
# ─────────────────────────────────────────────────────────────────────────────

spark.sql(f"""
    SELECT
        u.nombre            AS usuario,
        r.nombre_rol        AS rol_en_la_fecha,
        urh.fecha_inicio    AS inicio_rol,
        urh.fecha_fin       AS fin_rol
    FROM usuario_rol_historico urh
    JOIN dim_usuario u ON u.id_usuario = urh.id_usuario
    JOIN dim_rol     r ON r.id_rol     = urh.id_rol
    WHERE u.nombre = 'María García'
      AND urh.fecha_inicio <= DATE('{FECHA_CONSULTA}')
      AND (urh.fecha_fin IS NULL OR urh.fecha_fin >= DATE('{FECHA_CONSULTA}'))
""").show(truncate=False)

### 4.2 Cadena completa: ¿Con qué esquema, evaluaciones y metas trabajó María en abril 2024?

In [ ]:
spark.sql("""
    SELECT
        u.nombre                     AS usuario,
        r.nombre_rol                 AS rol,
        reh.nombre_esquema           AS esquema,
        e.nombre_evaluacion          AS evaluacion,
        eeh.peso * 100               AS peso_pct,
        p.nombre_periodo             AS periodo,
        meh.valor_meta               AS meta,
        meh.valor_minimo             AS minimo,
        e.unidad_medida              AS unidad

    FROM dim_usuario u

    -- Rol vigente en abril 2024
    JOIN usuario_rol_historico urh
        ON  urh.id_usuario    = u.id_usuario
        AND urh.fecha_inicio <= DATE('2024-04-30')
        AND (urh.fecha_fin IS NULL OR urh.fecha_fin >= DATE('2024-04-01'))

    -- Rol → Esquema vigente en abril 2024
    JOIN dim_rol r ON r.id_rol = urh.id_rol

    JOIN rol_esquema_historico reh
        ON  reh.id_rol        = urh.id_rol
        AND reh.fecha_inicio <= DATE('2024-04-30')
        AND (reh.fecha_fin IS NULL OR reh.fecha_fin >= DATE('2024-04-01'))

    -- Esquema → Evaluaciones con peso
    JOIN esquema_evaluacion_historico eeh
        ON  eeh.id_esquema    = reh.id_esquema
        AND eeh.activo        = TRUE
        AND (eeh.fecha_fin IS NULL OR eeh.fecha_fin >= DATE('2024-04-01'))

    JOIN dim_evaluacion e ON e.id_evaluacion = eeh.id_evaluacion

    -- Metas del período Abril 2024
    JOIN dim_periodo p
        ON  p.anio = 2024 AND p.mes = 4

    LEFT JOIN meta_evaluacion_historico meh
        ON  meh.id_esquema    = reh.id_esquema
        AND meh.id_evaluacion = eeh.id_evaluacion
        AND meh.id_periodo    = p.id_periodo

    WHERE u.nombre = 'María García'
    ORDER BY eeh.peso DESC
""").show(truncate=False)

### 4.3 Auditoría: historial completo de cambios de rol de todos los usuarios

In [ ]:
spark.sql("""
    SELECT
        u.nombre         AS usuario,
        r.nombre_rol     AS rol,
        urh.fecha_inicio,
        COALESCE(CAST(urh.fecha_fin AS STRING), 'Vigente') AS fecha_fin,
        reh.nombre_esquema
    FROM usuario_rol_historico urh
    JOIN dim_usuario u            ON u.id_usuario = urh.id_usuario
    JOIN dim_rol r                ON r.id_rol     = urh.id_rol
    LEFT JOIN rol_esquema_historico reh
        ON  reh.id_rol        = urh.id_rol
        AND reh.fecha_inicio <= urh.fecha_inicio
        AND (reh.fecha_fin IS NULL OR reh.fecha_fin >= urh.fecha_inicio)
    ORDER BY u.nombre, urh.fecha_inicio
""").show(truncate=False)

### 4.4 Validación de integridad del modelo

In [ ]:
print("=" * 60)
print("VALIDACIÓN DE INTEGRIDAD DEL MODELO")
print("=" * 60)

# 1 · Suma de pesos por esquema
print("\n[1] Suma de pesos por esquema (debe ser 1.0):")
spark.sql("""
    SELECT id_esquema, ROUND(SUM(peso), 4) AS suma_pesos,
           CASE WHEN ABS(SUM(peso) - 1.0) < 0.0001 THEN '✅ OK' ELSE '❌ Error' END AS estado
    FROM esquema_evaluacion_historico
    WHERE activo = TRUE AND fecha_fin IS NULL
    GROUP BY id_esquema
""").show()

# 2 · Usuarios sin asignación activa
print("[2] Usuarios sin rol activo (deberían ser 0):")
spark.sql("""
    SELECT u.nombre AS usuario_sin_rol_activo
    FROM dim_usuario u
    WHERE u.activo = TRUE
      AND NOT EXISTS (
          SELECT 1 FROM usuario_rol_historico urh
          WHERE urh.id_usuario = u.id_usuario
            AND urh.fecha_fin IS NULL
      )
""").show()

# 3 · Roles sin esquema activo
print("[3] Roles sin esquema activo:")
spark.sql("""
    SELECT r.nombre_rol AS rol_sin_esquema
    FROM dim_rol r
    WHERE r.activo = TRUE
      AND NOT EXISTS (
          SELECT 1 FROM rol_esquema_historico reh
          WHERE reh.id_rol    = r.id_rol
            AND reh.fecha_fin IS NULL
            AND reh.activo    = TRUE
      )
""").show()

print("\n✅ Notebook 03 completado — modelo de Configuración Histórica listo.")